In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE = "workspace.s4lake_bronze"
SILVER = "workspace.s4lake_silver"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

kna1 = spark.table(f"{BRONZE}.kna1")
knb1 = spark.table(f"{BRONZE}.knb1")

# 1. Deduplicação: numera os registros de cada cliente e fica com o primeiro
janela = Window.partitionBy("KUNNR").orderBy(
    (F.col("NAME1") != F.trim("NAME1")).asc(),  # prefere o registro sem espaços sobrando
    F.col("_ingestao_em").desc(),               # em caso de empate, o mais recente
)
kna1_unico = (
    kna1.withColumn("_ordem", F.row_number().over(janela))
    .filter("_ordem = 1")
    .drop("_ordem")
)

# 2. Junção com os dados da empresa, tipagem e renomeação
clientes = (
    kna1_unico.alias("g")
    .join(knb1.alias("e"), on="KUNNR", how="left")
    .select(
        F.col("KUNNR").alias("cod_cliente"),
        F.trim("g.NAME1").alias("razao_social"),
        F.when(F.trim("g.STCD1") == "", None).otherwise(F.trim("g.STCD1")).alias("cnpj"),
        F.col("g.ORT01").alias("cidade"),
        F.col("g.REGIO").alias("uf"),
        F.col("g.BRSCH").alias("ramo_atividade"),
        F.to_date("g.ERDAT", "yyyyMMdd").alias("data_cadastro"),
        F.col("e.ZTERM").alias("cond_pagamento"),
        F.substring("e.ZTERM", 2, 3).cast("int").alias("prazo_pagamento_dias"),
    )
    .withColumn("cnpj_ausente", F.col("cnpj").isNull())
)

clientes.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{SILVER}.clientes")

In [0]:
spark.sql(f"""
    SELECT count(*)                    AS linhas,
           count(DISTINCT cod_cliente) AS clientes_unicos,
           sum(CAST(cnpj_ausente AS INT)) AS sem_cnpj
    FROM {SILVER}.clientes
""").show()